### Initialisation

In [0]:
%sql
USE CATALOG dev_hcahps2026;

### Transform hospital_beds data

In [0]:
%sql
-- Create silver.hospital_beds table
DROP TABLE IF EXISTS dev_hcahps2026.silver.hospital_beds;
CREATE TABLE IF NOT EXISTS dev_hcahps2026.silver.hospital_beds AS
WITH hospital_beds_prep AS
(
-- Change data types, remove trailing spaces and adjust provider_ccn to 6 numbers 
SELECT LPAD(TRIM(provider_ccn), 6, '0') as provider_ccn,
    TRIM(hospital_name) as hospital_name,
    TRY_TO_DATE(TRIM(fiscal_year_begin_date)) as fiscal_year_begin_date,
    TRY_TO_DATE(TRIM(fiscal_year_end_date)) as fiscal_year_end_date,
    TRY_CAST(TRIM(number_beds) AS INT) as number_beds,
    ROW_NUMBER() OVER (PARTITION BY LPAD(TRIM(provider_ccn), 6, '0') ORDER BY TRY_TO_DATE(TRIM(fiscal_year_end_date)) DESC) as rn   
FROM dev_hcahps2026.bronze.hospital_beds
WHERE number_beds IS NOT NULL
)
SELECT *   
FROM hospital_beds_prep
WHERE rn = 1;


### Transform hcahps data

In [0]:
%sql
-- Create silver.hcahps_data_prep table
DROP TABLE IF EXISTS dev_hcahps2026.silver.hcahps_data_prep;
CREATE TABLE IF NOT EXISTS dev_hcahps2026.silver.hcahps_data_prep AS
SELECT *
FROM dev_hcahps2026.bronze.hcahps_data
-- We are only interested in cases when the question is "always true" and the hospital gets a rating above 9
WHERE CONTAINS (hcahps_question, "Always")
    OR CONTAINS (hcahps_question, "9");

-- Enable column mapping
ALTER TABLE dev_hcahps2026.silver.hcahps_data_prep
SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name'); 

-- Rename facility_id column
ALTER TABLE dev_hcahps2026.silver.hcahps_data_prep
RENAME COLUMN facility_id TO provider_ccn;

-- Replace "Not Available" values by 0
UPDATE dev_hcahps2026.silver.hcahps_data_prep
SET hcahps_answer_percent = CASE WHEN hcahps_answer_percent = 'Not Available' THEN '0' ELSE hcahps_answer_percent END,
    num_completed_surveys = CASE WHEN num_completed_surveys = 'Not Available' THEN '0' ELSE num_completed_surveys END,
    survey_response_rate_percent = CASE WHEN survey_response_rate_percent = 'Not Available' THEN '0' ELSE survey_response_rate_percent END;

-- Create silver.hcahps_data table
DROP TABLE IF EXISTS dev_hcahps2026.silver.hcahps_data;
CREATE TABLE IF NOT EXISTS dev_hcahps2026.silver.hcahps_data AS
-- Change data types and date format, remove trailing spaces 
SELECT TRIM(facility_name) as facility_name,
    TRIM(address) as address,
    TRIM(city) as city,
    TRIM(state) as state,
    TRIM(zipcode) as zipcode,
    TRIM(county_parish) as county_parish,
    TRIM(phone_number) as phone_number,
    TRIM(hcahps_measure_id) as hcahps_measure_id,
    TRIM(hcahps_question) as hcahps_question,
    TRIM(hcahps_answer_description) as hcahps_answer_description,
    TRY_CAST(TRIM(hcahps_answer_percent) AS INT) as hcahps_answer_percent,
    TRY_CAST(TRIM(num_completed_surveys) AS INT) as num_completed_surveys,
    TRY_CAST(TRIM(survey_response_rate_percent) AS INT) as survey_response_rate_percent,
    TRY_TO_DATE(TRIM(start_date), 'MM/dd/yyyy') as start_date,
    TRY_TO_DATE(TRIM(end_date)) as end_date,
-- Remove '.0' at the end of provider_ccn and adjust to 6 numbers
    LPAD(SPLIT_PART(TRIM(provider_ccn), '.', 1), 6, '0') as provider_ccn
FROM dev_hcahps2026.silver.hcahps_data_prep

